### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="clock_protein_toxicity",
    dataset_year="2021",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C59313",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder. 

mkdir -p local-data-warehouse/clock_protein_toxicity && wget -P local-data-warehouse/clock_protein_toxicity/ https://archive.ics.uci.edu/static/public/728/toxicity-2.zip && unzip local-data-warehouse/clock_protein_toxicity/toxicity-2.zip -d local-data-warehouse/clock_protein_toxicity/ && rm local-data-warehouse/clock_protein_toxicity/toxicity-2.zip
""",
    # References
    academic_reference_bibtex=r"""@article{gul2021structure,
  title={Structure-based design and classifications of small molecules regulating the circadian rhythm period},
  author={Gul, Seref and Rahim, Fatih and Isin, Safak and Yilmaz, Fatma and Ozturk, Nuri and Turkay, Metin and Kavakli, Ibrahim Halil},
  journal={Scientific reports},
  volume={11},
  number={1},
  pages={18510},
  year={2021},
  publisher={Nature Publishing Group UK London}
}
""",
    academic_reference_bibtex_key="gul2021structure",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We remove duplicated columns (same values for all rows).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Toxic",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Toxic",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "data.csv")

target_feature = "Toxic"
df = df.rename(columns={"Class": target_feature})
df[target_feature] = df[target_feature].astype("category")

# drop duplicated
duplicate_columns = [
    "nT10Ring", "nssS", "nssCH2", "khs.dsCH", "nHdsCH", "minHdNH", "SHdNH",
    "EE_DzZ", "SsBr", "maxsBr", "nF12HeteroRing", "n7Ring", "n7HeteroRing",
    "nT7Ring", "khs.sF", "nsF", "nF8HeteroRing", "nT8Ring", "nT8HeteroRing",
    "ndO", "nsCH3", "nHssNH", "nssNH", "nBr", "khs.sBr", "maxsCl",
    "khs.aaCH", "naaCH", "maxssssNp", "SaaS", "khs.sOH", "nsOH",
    "nT11HeteroRing", "maxdS", "mindS", "maxsNH2", "nssssC", "nT5Ring",
    "nT12Ring", "SpMax_Dzi", "khs.sNH2", "nsNH2", "SpMax_Dze", "minaaNH",
    "SpMax_Dzv", "nTG12HeteroRing", "nsOm", "minHsNH2", "minHtCH",
    "ntsC", "nBondsT", "khs.tsC", "nT5HeteroRing", "maxHaaNH", "ndNH",
    "nHdNH", "SdNH", "mindNH", "SpMax_Dzp", "EE_D", "naaS", "nsssN",
    "maxdCH2", "SdCH2", "naaNH", "khs.tN", "nssO", "maxtCH", "khs.dsN",
    "SHdCH2", "maxHdCH2", "SpMax_Dzs", "n6HeteroRing", "nF9Ring",
    "nT11Ring", "nT10HeteroRing", "nssssNp", "SpMax_Dzm", "SssS",
    "naaO", "khs.ddssS", "khs.sCl", "nsCl", "nsssCH", "nT9HeteroRing",
    "nFG12Ring"
]
df = df.drop(columns=duplicate_columns)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()